
## 5. Total consistency와 날짜 검증
- completed source total:
- category total:
- monthly total:
- customer total:
- 총합 일치 여부:
- completed 주문 날짜 오류 건수:
- 날짜 오류 영향 금액:

### 불일치가 있었다면 원인

## 6. 대표 시각화
![대표 그래프 1](images/graph01.png)
![대표 그래프 2](images/graph02.png)

### 그래프 선택 이유

### 그래프와 원본 집계값 일치 여부

### 그래프에서 직접 관찰한 사실

### 그래프만으로 말할 수 없는 것

## 7. 개인정보 검증
- 공개 고객 CSV 컬럼:
- 원본 `customer_id` 포함 여부:
- 이름/이메일/전화번호/주소 포함 여부:
- 익명 라벨 방식:
- PASS/FAIL:

### 나의 판단

## 8. 최종 Validation
`reports/ch08_project_validation.csv`의 결과를 요약합니다.

| 검증 항목 | 결과 | 내가 확인한 근거 |
| --- | --- | --- |
| pk_integrity |  |  |
| fk_integrity |  |  |
| merge_checks_pass |  |  |
| line_total_consistency |  |  |
| completed_total_consistency |  |  |
| category_sales_ratio_pct_sum |  |  |
| completed_rows_with_invalid_order_date |  |  |
| public_customer_columns_safe |  |  |

### FAIL이 있었다면 수정 내용

## 9. LLM 활용 기록
- 사용 여부:
- 사용 목적:
- Safe Context:
- Prompt 요약:
- 제안 요약:
- 반영/수정/보류:
- 사람이 검증한 근거:

## 10. 프로젝트 재현 확인
- `python scripts/run_midterm_project.py` 실행 여부:
- 재실행 결과:
- Notebook과 핵심 수치 일치 여부:
- 생성된 핵심 Evidence 파일 확인 여부:

![재실행 결과](images/step06_reproduce.png)

### 나의 해석과 판단
재실행 가능한 프로젝트가 왜 중요한지 작성하세요.

## 11. 최종 프로젝트 요약
### 핵심 인사이트 3개
1.
2.
3.

### 가장 중요한 업무·분석적 의미

### 현재 분석의 한계

### 다음 분석 제안
1.
2.
3.

## 최종 체크
- [ ] 질문과 지표가 연결됩니다.
- [ ] 원본 데이터에서 다시 시작했습니다.
- [ ] PK/FK·병합 검증 근거가 있습니다.
- [ ] `line_total` 계산 관계를 확인했습니다.
- [ ] 금액성 분석은 completed 범위를 사용했습니다.
- [ ] category/month/customer 총합이 source total과 일치합니다.
- [ ] 날짜 오류를 확인했습니다.
- [ ] 공개 고객 결과에서 원본 ID와 직접 식별정보를 제거했습니다.
- [ ] 대표 그래프와 원본 집계값을 비교했습니다.
- [ ] 원인 과대 해석이 없습니다.
- [ ] 최종 Validation이 모두 PASS입니다.
- [ ] 전체 프로젝트를 재실행했습니다.
- [ ] 한계와 다음 분석을 작성했습니다.
- [ ] 최종 Notebook URL을 제출합니다.

# Chapter 08 답안 양식. 작은 데이터 분석 프로젝트 완성하기

> 이 내용을 `chapter08.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026.09.17
- 최종 Notebook URL: https://github.com/sangjae-lee97/kant-axagent-study/blob/main/llm-data-analysis-course/chapter08/chapter08.ipynb

## 1. 프로젝트 질문
### 분석 질문
카테고리별 completed 주문 기준 금액은 어떻게 다른가?
### 분석 범위
- 주문 상태가 completed인 주문만 분석한다.
- completed 주문에 포함된 주문 항목(order_items)을 대상으로 한다.
- 상품(products)의 카테고리 정보를 연결하여 카테고리별로 집계한다.
- 카테고리별 주문 수량과 주문 금액을 비교한다.
### 사용할 데이터와 지표
order_status, category, quantity, unit_price
### 계산 기준
- `order_status == "completed"` 적용 여부:
- `line_total = quantity × unit_price` 확인 여부:

### 완료 기준
- completed 주문만 정상적으로 필터링되었는지 확인한다.
- orders와 order_items가 order_id 기준으로 정상 연결되었는지 확인한다.
- products가 product_id 기준으로 정상 연결되었는지 확인한다.
- line_total이 quantity × unit_price와 일치하는지 확인한다.
- 카테고리별 total_quantity를 계산한다.
- 카테고리별 total_sales를 계산한다.
- total_sales 기준으로 카테고리를 비교할 수 있는 결과표를 생성한다.

In [2]:
# 경로 설정
from course_utils.paths import get_project_root, get_data_dir
import pandas as pd
import numpy as np

print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())

RAW_DIR = get_project_root() / "data" / "raw"
PROCESSED_DIR = get_project_root() / "data" / "processed"
REPORT_DIR = get_project_root() / "reports"
PROJECT_ROOT = get_project_root()

프로젝트 루트 : C:\dev\kant-axagent-study\llm-data-analysis-course
데이터 폴더 : C:\dev\kant-axagent-study\llm-data-analysis-course\data


## 2. 입력 데이터와 전처리 검증
- 사용 원본 파일:
data/raw/customers.csv
data/raw/products.csv
data/raw/orders.csv
data/raw/order_items.csv
- shape, 핵심 결측/중복 결과:
customers
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
missing: 0
duplicates: 0

orders
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
missing: 0
duplicates: 0

order_items
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
missing: 0
duplicates: 0

prodcuts
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']
missing: 0
duplicates: 0
- 전처리 전후 변화:

![입력 데이터 검증](images/step02_data_validation.png)

### 나의 해석과 판단
현재 결측치, 전체 중복 행 없음
### 한계와 추가 확인 사항
id 중복은 있을 수 있음

In [5]:
customers = pd.read_csv(get_data_dir()/"raw"/"customers.csv")
orders = pd.read_csv(get_data_dir()/"raw"/"orders.csv")
order_items = pd.read_csv(get_data_dir()/"raw"/"order_items.csv")
products = pd.read_csv(get_data_dir()/"raw"/"products.csv")

In [9]:
raw_data = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "products": products,
}

In [6]:
for name, df in raw_data.items():
    print(name)
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print("missing:", int(df.isna().sum().sum()))
    print("duplicates:", int(df.duplicated().sum()))
    print()

customers
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
missing: 0
duplicates: 0

orders
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
missing: 0
duplicates: 0

order_items
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
missing: 0
duplicates: 0

prodcuts
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']
missing: 0
duplicates: 0



공통 전처리 로직

In [10]:
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)
processed_data = preprocess_sales_data(raw_data)
preprocessing_comparison = compare_shapes(
    raw_data,
    processed_data,
)
relationship_checks = validate_relationships(
    processed_data
)

## 3. PK/FK·병합 검증
### PK 결과
- 결측: 0
- 중복: 0
- PASS/FAIL: PASS

### FK 결과
- 미매칭: 0
- PASS/FAIL: PASS

### 병합 결과
- validate 관계: order_items : orders = 1 : N
- 병합 전 행 수: 764
- 병합 후 행 수: 764
- 미매칭: 0
- PASS/FAIL: PASS

### 나의 해석과 판단
전처리 완료 후 병합 성공

In [13]:
# PK 결측, 중복 확인
key_checks = []
for dataset, key in {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}.items():
    df = processed_data[dataset]
    key_checks.append({
        "dataset": dataset,
        "key": key,
        "missing_count": int(df[key].isna().sum()),
        "duplicate_count": int(df[key].duplicated().sum()),
    })
key_checks = pd.DataFrame(key_checks)
display(key_checks)

,dataset,key,missing_count,duplicate_count
0,customers,customer_id,0,0
1,products,product_id,0,0
2,orders,order_id,0,0
3,order_items,order_item_id,0,0


In [14]:
# FK 미매칭 확인
missing_customer_ids = set(
    processed_data["orders"]["customer_id"].dropna()
) - set(
    processed_data["customers"]["customer_id"].dropna()
)
print("orders → customers 미매칭:", len(missing_customer_ids))

orders → customers 미매칭: 0


In [15]:
order_sales = order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
print(order_sales)

     order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0                1         1         100         3      102000          123   
1                2         1          87         5       25000          123   
2                3         1           7         3      142000          123   
3                4         1           9         3      193000          123   
4                5         2          72         4      189000           77   
..             ...       ...         ...       ...         ...          ...   
759            760       297          42         4       28000           22   
760            761       298          40         4      174000           64   
761            762       299           8         2      189000          135   
762            763       299          12         4      175000          135   
763            764       300          59         5       32000           96   

     order_date order_status _merge  
0    2026-05-

In [16]:
print("병합 전 행 수:", len(order_items))
print("병합 후 행 수:", len(order_sales))
print(
    "미매칭:",
    int(order_sales["_merge"].eq("left_only").sum()),
)

병합 전 행 수: 764
병합 후 행 수: 764
미매칭: 0


In [23]:
order_sales["line_total"] = (
    order_sales["quantity"] * order_sales["unit_price"]
)

In [25]:
# line_total 행 추가 및 검증
expected_line_total = (
    order_items["quantity"] * order_items["unit_price"]
)
line_total_matches = np.isclose(
    order_sales["line_total"],
    expected_line_total,
)
print("일치:", int(line_total_matches.sum()))
print("불일치:", int((~line_total_matches).sum()))



일치: 764
불일치: 0


In [ ]:
# completetd 상태인 것만 뽑기
completed_order_sales = order_sales.loc[
    order_sales["order_status"].eq("completed")
].copy()
print(completed_order_sales)

     order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0                1         1         100         3      102000          123   
1                2         1          87         5       25000          123   
2                3         1           7         3      142000          123   
3                4         1           9         3      193000          123   
12              13         6          83         3       24000           87   
..             ...       ...         ...       ...         ...          ...   
757            758       296          40         1      174000          116   
758            759       297          20         5       80000           22   
759            760       297          42         4       28000           22   
761            762       299           8         2      189000          135   
762            763       299          12         4      175000          135   

     order_date order_status _merge  
0    2026-05-

## 4. 핵심 EDA 결과
### 결과 1
카테고리별 completed 주문 기준 금액
- 수치/표:
- 결과 관찰:
- 나의 해석과 판단:
- 업무·분석적 의미:
- 한계:

### 결과 2
월별 completed 주문 기준 금액과 주문 수
- 수치/표:
- 결과 관찰:
- 나의 해석과 판단:
- 업무·분석적 의미:
- 한계:

### 결과 3
completed 주문 기준 고객별 구매 금액
- 수치/표:
- 결과 관찰:
- 나의 해석과 판단:
- 업무·분석적 의미:
- 한계:

In [ ]:
# 카테고리별
category_sales =(
    compl
)